In [1]:
from openai import OpenAI
import os
client = OpenAI(
    base_url= "https://openrouter.ai/api/v1",
    api_key= os.getenv("OPEN_API_KEY")
)

In [2]:
import pdfplumber

def extract_text(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

In [3]:
# def chunk_text(text, chunk_size=500):
#     return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i+chunk_size])
    return chunks

In [4]:
def get_embeddings(text):
    emb = client.embeddings.create(
        model= "text-embedding-3-small",
        input = text
    )
    return emb.data[0].embedding

In [5]:
text = extract_text("parking.pdf")
chunk_texts = chunk_text(text, chunk_size=500)
chunk_embeddings = []
for chunk in chunk_texts:
    chunk_embeddings.append(get_embeddings(chunk))


In [47]:
import faiss
import numpy as np

dimension = len(chunk_embeddings[0])

index = faiss.IndexFlatL2(dimension)

vectors = np.array(chunk_embeddings).astype("float32") #fais uses float32 unlike python dfaul float64

index.add(vectors)

In [30]:

extract_prompt = """
Extract the following information from the paper in strict JSON format:

{
 "authors": "",
 "year": "",
 "title": "",
 "objectives": "",
 "methods": "",
 "results": "",
 "limitations": ""
}

If information is missing, return null.
Do not add commentary.
"""

summary_prompt = """ 
Using the structured data below, write exactly one cohesive academic paragraph under 200 words.

Use academic transition phrases (e.g.,'John Doe et al. (2001) proposed....' , 'The primary goal of (John Doe et al. 2001)...','The system aims to...', 'The system uses...',  'The study explores...', 'Utilizing a methodology of...', 'The findings reveal...', 'The major drawback...' and other phrases) to ensure a fluid narrative and include any other critical unique insights found in the paper.
If null, do not invent data to fill
No bullet points.
No line breaks.

Structured Data:
{json_output}
"""

extract = client.chat.completions.create(
    model = "meta-llama/llama-3.3-70b-instruct",
    temperature=0.1,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": extract_prompt},
        {"role": "user", "content": text}
    ]
)

json_output = extract.choices[0].message.content
review = client.chat.completions.create(
    model = "meta-llama/llama-3.3-70b-instruct",
    temperature=0.1,
    messages=[
        {"role": "system", "content": summary_prompt},
        {"role": "user", "content": json_output}
    ]
)

print(review.choices[0].message.content)

Amira A. Elsonbaty and Mahmoud Shams (2020) proposed a smart parking management system, with the primary goal of providing a solution to the parking problem, reducing the time spent searching for parking lots, and eliminating unnecessary vehicle travel. The system aims to achieve this through a set of commands within the Arduino, utilizing hardware components such as Arduino, GSM Module, and Node MCU, among others. The study explores the potential benefits of this system, which include avoiding time wastage, reducing pollution, and minimizing fuel consumption. The findings reveal that users can book a car park for up to 24 hours, highlighting the system's potential to streamline parking processes and mitigate environmental impacts. Overall, the system uses a combination of hardware and software components to provide an efficient and user-friendly parking management solution.


In [49]:
user_question = "what are the stages involved in finding a parking space"
query_embedding = get_embeddings(user_question)
query_vector = np.array([query_embedding]).astype("float32")

k = 5
distances, indices = index.search(query_vector, k)

retrieved_chunks = [chunk_texts[i] for i in indices[0]]

print("----- RETRIEVED CHUNKS -----")
for chunk in retrieved_chunks:
    print(chunk)
    print("\n----------------------------\n")
    
context = "\n\n".join(retrieved_chunks)

prompt = f"""
Answer the question using ONLY the context below.
If answer is not in context, say "Not included in the research paper"


Context:
{context}

Question:
{user_question}
"""

response = client.chat.completions.create(
    model = "meta-llama/llama-3.3-70b-instruct",
    temperature=0.1,
    messages=[
        {"role": "system", "content": "You answer questions strictly from provided context."},
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

----- RETRIEVED CHUNKS -----
cognize the nearest available parking
zone. So, the main purpose of the system is to provide a solution to the parking problem, to
reduce the time to search for parking lots, and to eliminate unnecessary travel for vehicles
4.3. How Smart Parking Works
Smart parking suggests an IoT-based system that sends data to free and busy parking places via
net/mobile applications. The IoT-network includes sensors and microcontrollers, which are found
in each parking place. We implemented an enclosed smart 

----------------------------

used in Smart parking [9], the Android application map forward data of the empty place of the
user. Each slot had an LED for finding out the parking space and booking. The Infrared sensor in
[10] was implemented to find out a free place and open the entry and exit gate. RFID tag issued
to approve an individual's entry to the parking place using the mobile application the ACO
algorithm was presented in [11] to calculate the shortest pat